In [52]:
# ## 1. Imports & configuración

# %%
from pathlib import Path
import pandas as pd, numpy as np, math, re
from collections import defaultdict

# Carpeta con las escenas ya a 1 Hz
DATA_DIR = Path("datos_sliding")

# Anchors en píxeles (mismo sistema que la imagen)
ANCHORS_PX = {
    "Raspy1": (644, 452),
    "Raspy2": (635,  55),
    "Raspy3": ( 68, 643),
    "Raspy4": ( 70, 100),
}

PX_PER_CM = 550 / 380   # ≈ 1.4474 px = 1 cm

# Ground‑truth puntos fijos A‑D en píxeles
TRUTH_POINTS = {
    "A": (100, 500),
    "B": (122, 223),
    "C": (370, 490),
    "D": (283, 179),
}

# Horarios GT por escena "mm:ss" (inicio, fin, etiqueta)
GT_SCHEDULES = {
    "Escena1": [
        ("1:02", "1:20", "D"),
        ("1:21", "1:40", "C"),
        ("1:41", "1:55", "A"),
        ("1:56", "2:05", "B"),
    ],
    "Escena2": [
        ("2:36", "2:56", "B"),
        ("2:57", "3:06", "A"),
        ("3:07", "3:27", "C"),
        ("3:28", "3:36", "D"),
    ],
    "Escena3": [
        ("11:55", "12:10", "C"),
        ("12:11", "12:25", "D"),
        ("12:26", "12:36", "B"),
        ("12:37", "12:55", "A"),
    ],
    "Escena4": [
        ("13:13", "13:34", "A"),
        ("13:35", "13:49", "B"),
        ("13:50", "14:04", "C"),
        ("14:05", "14:13", "D"),
    ],
    "Escena5": [
        ("14:59", "15:15", "B"),
        ("15:16", "15:35", "C"),
        ("15:36", "15:50", "A"),
        ("15:51", "15:59", "D"),
    ],
    "Escena6": [
        ("16:20", "16:41", "D"),
        ("16:42", "16:56", "B"),
        ("16:57", "17:10", "C"),
        ("17:11", "17:20", "A"),
    ],
}

VERBOSE = False   # True → imprime error por timestamp


In [53]:
# %%  funciones auxiliares
def read_meas_file(path: Path) -> pd.DataFrame:
    """Lee TSV 1 Hz (sin cabecera) y devuelve DataFrame limpio."""
    df = pd.read_csv(path, sep="\t", header=None,
                     names=["timestamp", "distance_cm", "rssi"])
    # tipos
    df["timestamp"]    = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
    df["distance_cm"]  = pd.to_numeric(df["distance_cm"], errors="coerce")
    df["rssi"]         = pd.to_numeric(df["rssi"], errors="coerce")
    return df.dropna().reset_index(drop=True)

def build_gt_lookup(schedule):
    """Convierte [('m:ss', 'm:ss', 'A'), …] → [(sec0,sec1,(x,y)), …]."""
    lut = []
    for start, end, label in schedule:
        m0, s0 = map(int, start.split(":"))
        m1, s1 = map(int, end.split(":"))
        sec0, sec1 = m0*60 + s0, m1*60 + s1
        lut.append((sec0, sec1, TRUTH_POINTS[label]))
    return lut

def gt_for_sec(sec, lut):
    """Devuelve punto GT para el segundo relativo `sec`."""
    for t0, t1, pt in lut:
        if t0 <= sec <= t1:
            return pt
    return lut[-1][2]      # si cae fuera, usa el último punto

def multilaterate_2d(rngs_cm, anchors_cm):
    """Mínimos cuadrados lineal 2-D con ≥3 anclas."""
    ids = list(rngs_cm.keys())
    if len(ids) < 3:
        raise ValueError("<3 rangos")
    x1, y1 = anchors_cm[ids[0]]
    d1     = rngs_cm[ids[0]]
    A, b = [], []
    for aid in ids[1:]:
        xi, yi = anchors_cm[aid]
        di     = rngs_cm[aid]
        A.append([2*(xi-x1), 2*(yi-y1)])
        b.append(d1**2 - di**2 + xi**2 - x1**2 + yi**2 - y1**2)
    A = np.array(A); b = np.array(b)
    pos, *_ = np.linalg.lstsq(A, b, rcond=None)
    return pos  # (x_cm, y_cm)

def load_scene(scene_dir: Path) -> pd.DataFrame:
    """Concatena los TSV 1 Hz de las cuatro Raspy y devuelve un DataFrame único."""
    frames = []
    for idx in range(1, 5):
        file = scene_dir / f"{idx}.tsv"
        dfi  = read_meas_file(file)
        dfi["anchor_id"] = f"Raspy{idx}"
        frames.append(dfi)
    return pd.concat(frames, ignore_index=True)

In [54]:
# %%  ejecución y resumen
DEBUG = False            # True → prints por timestamp

px_per_cm  = PX_PER_CM
anchors_cm = {aid: (x/px_per_cm, y/px_per_cm) for aid, (x, y) in ANCHORS.items()}

scene_stats = {}   # name → (mae_px, mae_cm, n)

for scene_dir in sorted(DATA_DIR.iterdir()):
    if not scene_dir.is_dir() or not scene_dir.name.startswith("Escena"):
        continue
    name = scene_dir.name
    print(f"\n━━ {name} ━━━━━━━━━━━━━━━━━━━━━━━━━")

    df = load_scene(scene_dir)
    print("Lecturas totales:", len(df))

    grouped = defaultdict(dict)
    for _, row in df.iterrows():
        grouped[row["timestamp"]][row["anchor_id"]] = row["distance_cm"]

    lut = build_gt_lookup(GT_SCHEDULES[name])
    if not grouped:
        print("⚠️  Sin datos en la escena")
        continue
    t0 = min(grouped.keys())

    px_errs, cm_errs = [], []
    for ts, rngs in grouped.items():
        if len(rngs) < 3:
            continue
        sec_rel = ts - t0
        gt_px   = gt_for_sec(sec_rel, lut)

        try:
            est_cm = multilaterate_2d(rngs, anchors_cm)
        except Exception as e:
            if DEBUG:
                print("  ×", ts, e)
            continue

        est_px  = (est_cm[0]*px_per_cm, est_cm[1]*px_per_cm)
        err_px  = math.hypot(est_px[0]-gt_px[0], est_px[1]-gt_px[1])
        px_errs.append(err_px)
        cm_errs.append(err_px / px_per_cm)

        if DEBUG:
            print(f"  t={sec_rel:5.1f}s  err={err_px:6.2f}px "
                  f"({err_px/px_per_cm:6.2f} cm)")

    if px_errs:
        mae_px = sum(px_errs)/len(px_errs)
        mae_cm = sum(cm_errs)/len(cm_errs)
        scene_stats[name] = (mae_px, mae_cm, len(px_errs))
        print(f"MAE = {mae_px:.2f} px  /  {mae_cm:.2f} cm   (n={len(px_errs)})")
    else:
        print("⚠️  Sin estimaciones válidas")

# resumen global
if scene_stats:
    g_px = sum(v[0] for v in scene_stats.values())/len(scene_stats)
    g_cm = sum(v[1] for v in scene_stats.values())/len(scene_stats)

    print("\n═════ Resumen global ═════")
    for sc, (px, cm, n) in scene_stats.items():
        print(f"{sc:10}: {px:6.2f} px | {cm:6.2f} cm   (n={n})")
    print("----------------------------------")
    print(f"Global MAE: {g_px:.2f} px  /  {g_cm:.2f} cm")
else:
    print("😕  Ninguna escena produjo estimaciones.")


━━ Escena1 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 13816
MAE = 369.76 px  /  255.47 cm   (n=3735)

━━ Escena2 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 13807
MAE = 378.94 px  /  261.81 cm   (n=3586)

━━ Escena3 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 14304
MAE = 315.72 px  /  218.14 cm   (n=3583)

━━ Escena4 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 13669
MAE = 413.96 px  /  286.01 cm   (n=3570)

━━ Escena5 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 13724
MAE = 425.40 px  /  293.91 cm   (n=3401)

━━ Escena6 ━━━━━━━━━━━━━━━━━━━━━━━━━
Lecturas totales: 13875
MAE = 431.78 px  /  298.32 cm   (n=3574)

═════ Resumen global ═════
Escena1   : 369.76 px | 255.47 cm   (n=3735)
Escena2   : 378.94 px | 261.81 cm   (n=3586)
Escena3   : 315.72 px | 218.14 cm   (n=3583)
Escena4   : 413.96 px | 286.01 cm   (n=3570)
Escena5   : 425.40 px | 293.91 cm   (n=3401)
Escena6   : 431.78 px | 298.32 cm   (n=3574)
----------------------------------
Global MAE: 389.26 px  /  268.94 cm


In [55]:
from pathlib import Path
import pandas as pd

# --- Adjust these two lines ------------------------------------
SCENES_DIR = Path("datos_organizados_escenas")       # parent folder containing Escena1/, …
TSV_GLOB   = "*.tsv"                     # pattern of the per-anchor files
# ---------------------------------------------------------------

rows_per_scene = {}

for esc_dir in sorted(SCENES_DIR.glob("Escena*")):      # or "config*" if you prefer
    total_rows = 0
    for tsv in esc_dir.glob(TSV_GLOB):
        # count rows quickly without loading whole file into memory
        total_rows += sum(1 for _ in open(tsv, "rb"))
    rows_per_scene[esc_dir.name] = total_rows

# --- print a neat summary --------------------------------------
print("Scene     Rows")
print("──────────────")
for scene, n in rows_per_scene.items():
    print(f"{scene:8} {n:,}")

# Optional: save to CSV
# pd.Series(rows_per_scene, name="rows").to_csv("row_count_by_scene.csv")

Scene     Rows
──────────────
Escena1  10,492
Escena2  6,025
Escena3  4,620
Escena4  3,739
Escena5  2,821
Escena6  2,380
